In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

from pyspark.sql.window import Window

In [69]:
spark = SparkSession.builder.appName("AnalyseProcessed").getOrCreate()

In [68]:
spark.sparkContext.setLogLevel("ERROR")

In [54]:
from pathlib import Path

notebook_dir = Path().resolve()  # notebooks/
data_path = notebook_dir.parent / "data" / "processed"

df = spark.read.option("header", True).parquet(str(data_path))

df.show(7)

+---------+---------------------+--------------------+---------+---------------------+--------------------+-------------------+-------------+-----------------------+----------+--------------------+--------------------+----------+----------+---------------------+--------------------------------+---------------+----------+--------------------+--------------------+---------------------------+-------------------+------------+--------------------+--------------------+-------------------+------------+
|      ath|ath_change_percentage|            ath_date|      atl|atl_change_percentage|            atl_date| circulating_supply|current_price|fully_diluted_valuation|  high_24h|                  id|        last_updated|   low_24h|market_cap|market_cap_change_24h|market_cap_change_percentage_24h|market_cap_rank|max_supply|                name|    price_change_24h|price_change_percentage_24h|       total_supply|total_volume|              7d_avg|              7d_max|             7d_low|updated_date

The data below shows that some coins appear on more days than others which is a data quality issue. 

In [30]:
from pyspark.sql.functions import count

df.groupBy("updated_date").agg(count("name").alias("coin_count")).show()

+------------+----------+
|updated_date|coin_count|
+------------+----------+
|  2026-02-26|       101|
|  2026-03-05|       104|
|  2026-03-03|       100|
|  2026-03-08|       100|
|  2026-02-27|        99|
|  2026-03-06|        99|
|  2026-03-10|        99|
|  2026-03-02|        98|
|  2026-03-01|         1|
|  2026-03-09|        97|
|  2026-02-28|         1|
+------------+----------+



In [ ]:
# window_spec = Window.partitionBy("id").orderBy(desc("updated_date"))

# most_updated_data = (
#     df.withColumn("rn", row_number().over(window_spec)).filter("rn = 1").drop("rn")
# )

max_date = df.select(F.max("updated_date")).collect()[0][0]

recent_data = df.filter(F.col("updated_date") == max_date).select(
    "name", "current_price", "market_cap", "updated_date", "total_volume"
)

recent_data.show(5)

+--------------+-------------+----------+------------+------------+
|          name|current_price|market_cap|updated_date|total_volume|
+--------------+-------------+----------+------------+------------+
|Official Trump|          2.9| 674884595|  2026-03-10|1.14179145E8|
|      MemeCore|         1.48|2583172933|  2026-03-10|   8672976.0|
|      Filecoin|     0.936391| 712144794|  2026-03-10| 7.9409019E7|
|       HTX DAO|      1.58E-6|1452490977|  2026-03-10| 4.5031168E7|
|    PayPal USD|          1.0|4076924162|  2026-03-10|1.09004233E8|
+--------------+-------------+----------+------------+------------+
only showing top 5 rows


## Ranking current price

In [73]:
rank_price_spec = Window.partitionBy("updated_date").orderBy(desc("current_price"))

# curr_top_price = ranking_df.sort(desc("current_price")).show(10)

curr_top_price = recent_data.withColumn(
    "current_price_rank", rank().over(rank_price_spec)
)

curr_top_price.show(10)

+------------+-------------+-------------+------------+---------------+------------------+
|        name|current_price|   market_cap|updated_date|   total_volume|current_price_rank|
+------------+-------------+-------------+------------+---------------+------------------+
|     Bitcoin|      69908.0|1398271709508|  2026-03-10|5.1282369635E10|                 1|
|    PAX Gold|      5178.87|   2584030099|  2026-03-10|   2.81308729E8|                 2|
| Tether Gold|      5140.17|   2901219409|  2026-03-10|   4.28789746E8|                 3|
|    Ethereum|      2042.73| 246527256426|  2026-03-10|2.2232642628E10|                 4|
|         BNB|       643.74|  87810028829|  2026-03-10|   9.35680408E8|                 5|
|Bitcoin Cash|       446.51|   8937593910|  2026-03-10|   2.30673715E8|                 6|
|      Monero|       343.86|   6349027985|  2026-03-10|    8.7567025E7|                 7|
|       Zcash|       218.95|   3632787875|  2026-03-10|   3.06683167E8|                 8|

## Ranking Market Cap

In [77]:
rank_market_spec = Window.partitionBy("updated_date").orderBy(desc("market_cap"))


curr_top_market = recent_data.withColumn(
    "market_cap_rank", rank().over(rank_market_spec)
)

curr_top_market.show(10)

+------------+-------------+-------------+------------+---------------+---------------+
|        name|current_price|   market_cap|updated_date|   total_volume|market_cap_rank|
+------------+-------------+-------------+------------+---------------+---------------+
|     Bitcoin|      69908.0|1398271709508|  2026-03-10|5.1282369635E10|              1|
|    Ethereum|      2042.73| 246527256426|  2026-03-10|2.2232642628E10|              2|
|      Tether|     0.999992| 183915899778|  2026-03-10|7.9733958378E10|              3|
|         BNB|       643.74|  87810028829|  2026-03-10|   9.35680408E8|              4|
|         XRP|         1.38|  84752093433|  2026-03-10|  2.341866174E9|              5|
|        USDC|     0.999946|  78203696599|  2026-03-10|1.2129440044E10|              6|
|      Solana|        86.32|  49307798996|  2026-03-10|  4.243082534E9|              7|
|        TRON|     0.286038|  27102034375|  2026-03-10|   4.62277348E8|              8|
|Figure Heloc|        1.042|  16

## Aggregate Market Capitalization

In [8]:
df.select(
    "name", "current_price", "market_cap", "total_volume", "updated_date"
).createOrReplaceTempView("crypto_prices")

In [9]:
agg_market = spark.sql(
    """
    SELECT updated_date, sum(market_cap) as total_market_cap
    FROM crypto_prices 
    GROUP BY updated_date
    ORDER BY updated_date DESC
"""
)

agg_market.show()

+------------+----------------+
|updated_date|total_market_cap|
+------------+----------------+
|  2026-03-10|   2399896922025|
|  2026-03-09|   2321225427182|
|  2026-03-08|   2332237691268|
|  2026-03-06|   2421682587777|
|  2026-03-05|   2479865301301|
|  2026-03-03|   2350972043210|
|  2026-03-02|   2295054269100|
|  2026-03-01|     15908254518|
|  2026-02-28|      1883793196|
|  2026-02-27|   2354599981906|
|  2026-02-26|   2382486193657|
+------------+----------------+



## Average Price

In [83]:
avg_price = spark.sql(
    """
    SELECT name as Name, avg(current_price) as average_price
    FROM crypto_prices 
    GROUP BY name
    ORDER BY avg(current_price) DESC
    """
)

avg_price.show()

+--------------------+------------------+
|                Name|     average_price|
+--------------------+------------------+
|             Bitcoin| 68715.44444444444|
|            PAX Gold| 5216.723333333332|
|         Tether Gold| 5175.185555555556|
|            Ethereum| 2027.028888888889|
|                 BNB| 633.1533333333334|
|        Bitcoin Cash| 459.7755555555555|
|              Monero| 349.3300000000001|
|               Zcash|223.82555555555555|
|           Bittensor| 186.0822222222222|
|                OUSG|114.47111111111111|
|                Aave|114.36777777777777|
|                 OKB| 86.58333333333333|
|              Solana| 86.31555555555555|
|               Quant| 64.07333333333334|
|            Litecoin| 54.75222222222223|
|       WhiteBIT Coin|52.233333333333334|
|         Hyperliquid|30.926666666666662|
|              Decred| 30.26555555555556|
|Superstate Short ...|11.005555555555555|
|           Avalanche| 9.248888888888889|
+--------------------+------------

## Volume to Market Ratio

In [75]:
curr_top_market = recent_data.withColumn("rank", rank().over(rank_market_spec))

vol_market_ratio = (
    df.withColumn(
        "vol_market_ratio",
        F.when(
            F.col("market_cap") > 0,
            F.round(F.col("total_volume") / F.col("market_cap"), 5),
        ).otherwise(0.0),
    )
    .orderBy(F.col("vol_market_ratio").desc())
    .select(
        "name",
        "current_price",
        "market_cap",
        "total_volume",
        "updated_date",
        "vol_market_ratio",
    )
)

vol_market_ratio.show(5)

rank_vol_market_spec = Window.orderBy(desc("vol_market_ratio"))
window_spec = Window.partitionBy("name").orderBy(desc("vol_market_ratio"))

# gets the highest vol-to-market ratio for each coin and ranks them
vol_market_ratio_rank = (
    vol_market_ratio.withColumn("rn", row_number().over(window_spec))
    .filter("rn = 1")
    .drop("rn")
    .withColumn("vol_market_rank", rank().over(rank_vol_market_spec))
)

vol_market_ratio_rank.show(10)

+------+-------------+------------+----------------+------------+----------------+
|  name|current_price|  market_cap|    total_volume|updated_date|vol_market_ratio|
+------+-------------+------------+----------------+------------+----------------+
|  USD1|     0.999959|  4703542500|   4.050956274E9|  2026-02-27|         0.86126|
|  USD1|     0.999996|  4705551140|     3.7583704E9|  2026-02-26|         0.79871|
|Tether|          1.0|183912220526|1.11298241808E11|  2026-03-05|         0.60517|
|  USD1|     0.999117|  4615776647|   2.644099229E9|  2026-03-05|         0.57284|
|Tether|      0.99988|183606483628| 9.7552146231E10|  2026-03-03|         0.53131|
+------+-------------+------------+----------------+------------+----------------+
only showing top 5 rows
+-------------+-------------+------------+----------------+------------+----------------+---------------+
|         name|current_price|  market_cap|    total_volume|updated_date|vol_market_ratio|vol_market_rank|
+-------------+--

## Top Performing Assets 

find coins which meet the following criteria: 
- Market capitalization: the *higher* the *better* 
- Price: *higher* price means greater demand 
- Volume/Market Cap Ratio: determines stability of the coin. Dependent on other factors.

Therefore, the ranking is determined using weighted sums. 
1. Market cap contributes 40% → weight of `0.4`
2. Price contributes 40% → weight of `0.4`
3. Vol/market ratio contributes 20% → weight of `0.2`

In [ ]:
top_performing_assets = (
    curr_top_market.select("name", "market_cap", "market_cap_rank")
    .join(
        curr_top_price.select("name", "current_price", "current_price_rank"),
        on=["name"],
    )
    .join(
        vol_market_ratio_rank.select(
            "name", "total_volume", "vol_market_ratio", "vol_market_rank"
        ),
        on=["name"],
    )
)

top_performing_assets = top_performing_assets.withColumn(
    "top_performing_rank",
    round(
        top_performing_assets.market_cap_rank * 0.4
        + top_performing_assets.current_price_rank * 0.4
        + top_performing_assets.vol_market_rank * 0.2,
        0,
    ),
).orderBy("top_performing_rank")

top_performing_assets.show(10)

+------------+-------------+---------------+-------------+------------------+----------------+----------------+---------------+-------------------+
|        name|   market_cap|market_cap_rank|current_price|current_price_rank|    total_volume|vol_market_ratio|vol_market_rank|top_performing_rank|
+------------+-------------+---------------+-------------+------------------+----------------+----------------+---------------+-------------------+
|    Ethereum| 246527256426|              2|      2042.73|                 4| 3.1391592618E10|         0.12236|             25|                7.0|
|     Bitcoin|1398271709508|              1|      69908.0|                 1|  7.194525264E10|         0.04957|             52|               11.0|
|      Solana|  49307798996|              7|        86.32|                13|   6.426767762E9|         0.12365|             24|               13.0|
| Tether Gold|   2901219409|             34|      5140.17|                 3|    9.75957539E8|          0.3235| 